# 自动驾驶 - 汽车检测

欢迎来到第三周编程作业！在此 Notebook 中，你将使用非常强大的 YOLO 模型实现目标检测。这个Notebook中的许多想法在两篇YOLO论文中都有描述：[Redmon et al., 2016](https://arxiv.org/abs/1506.02640)和[雷蒙和法哈迪，2016](https://arxiv.org/abs/1612.08242)。 

**在本作业结束时，你将能够**：

- 检测汽车检测数据集中的对象
- 实现非极大值抑制以提高准确性
- 实现交并比
- 处理边界框，深度学习中流行的一种图像标注

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/b6TbJ/optional-downloading-your-notebook-downloading-your-workspace-and-refreshing) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 问题描述](#1)
- [2 - YOLO](#2)
    - [2.1 - 模型详细信息](#2-1)
    - [2.2 - 根据类别分数阈值筛选](#2-2)
        - [练习 1 - yolo_filter_boxes](#ex-1)
    - [2.3 - 非极大值抑制](#2-3)
        - [练习 2 - iou](#ex-2)
    - [2.4 - YOLO 非极大值抑制](#2-4)
        - [练习 3 - yolo_non_max_suppression](#ex-3)
    - [2.5 - 整合筛选过程](#2-5)
        - [练习 4 - yolo_eval](#ex-4)
- [3 - 在图像上测试 YOLO 预训练模型](#3)
    - [3.1 - 定义类别、锚框和图像形状](#3-1)
    - [3.2 - 加载预训练模型](#3-2)
    - [3.3 - 将模型的输出转换为可用的边界框张量](#3-3)
    - [3.4 - 过滤框](#3-4)
    - [3.5 - 在图像上运行 YOLO](#3-5)
- [4 - YOLO 总结](#4)
- [5 - 参考文献](#5)

<a name='0'></a>
## 依赖库

运行以下单元来加载包和依赖项，这些包和依赖项将在你构建目标检测器时派上用场！

In [ ]:
### v2.0

In [ ]:
import argparse
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import scipy.io
import scipy.misc
import numpy as np
import pandas as pd
import PIL
from PIL import ImageFont, ImageDraw, Image
import tensorflow as tf
from tensorflow.python.framework.ops import EagerTensor

from tensorflow.keras.models import load_model
from yad2k.models.keras_yolo import yolo_head
from yad2k.utils.utils import draw_boxes, get_colors_for_classes, scale_boxes, read_classes, read_anchors, preprocess_image

%matplotlib inline

<a name='1'></a>
## 1 - 问题描述

你正在开发一辆自动驾驶汽车。走吧你！作为该项目的关键组成部分，你希望首先构建一个汽车检测系统。为了收集数据，你在汽车的引擎盖（即前部）上安装了一个摄像头，它会在你行驶时每隔几秒钟拍摄一次前方道路的照片。 

<center>
<video width="400" height="200" src="nb_images/road_video_compressed2.mp4" type="video/mp4" controls>
</video>
</center>

<div style="text-align:center;">在硅谷行驶时用车载摄像头拍摄的照片。 <br> 数据集由 <a href="https://www.drive.ai/">drive.ai</a> 提供。</div>

你已将所有这些图像收集到一个文件夹中，并通过在你找到的每辆车周围绘制边界框来标记它们。以下是边界框的示例：

<img src="nb_images/box_label.png" style="width:500px;height:250px;">
<div style="text-align:center;"><u><b>图1</b></u>：盒子的定义<br></div>

如果你希望目标检测器识别 80 个类，你可以将类标签 $c$ 表示为 1 到 80 之间的整数，或者表示为 80 维向量（有 80 个数字），其中一个分量为 1，其余为 0。视频讲座使用后一种表示形式；在此 Notebook 中，你将使用两种表示法，具体取决于哪种表示法对于特定步骤更方便。  

本练习将介绍 YOLO（You Only Look Once）如何完成目标检测，并将它应用于汽车检测。训练 YOLO 模型需要大量计算，因此这里已为你加载预训练权重，可直接使用。 

<a name='2'></a>
## 2 - YOLO

YOLO（You Only Look Once）是一种广泛使用的算法，兼具较高准确率和实时运行能力。“只看一次”是指仅需让图像通过网络完成一次前向传播，即可进行预测。完成非极大值抑制后，模型会输出识别出的目标及其边界框。

<a name='2-1'></a>
### 2.1 - 模型详细信息

#### 输入和输出
- **输入**是一批图像，每个图像的形状为 (608, 608, 3)
- **输出**是边界框以及已识别的类的列表。每个边界框由 6 个数字 $(p_c, b_x, b_y, b_h, b_w, c)$ 表示，如上所述。如果将 $c$ 展开为 80 维向量，则每个边界框都由 85 个数字表示。 

#### 锚框（Anchor Boxes）
* 通过探索训练数据来选择锚框，以选择代表不同类别的合理的高/宽比。  对于这个作业，为你选择了5个锚框（覆盖80个类），并存储在文件“./model_data/yolo_anchors.txt”中
* 基于锚框的倒数第二个维度的编码张量的维度为$(m, n_H,n_W,anchors,classes)$。
* YOLO架构是：IMAGE (m, 608, 608, 3) -> DEEP CNN -> ENCODING (m, 19, 19, 5, 85)。  


#### 编码
让我们更详细地看看这个编码代表什么。 

<img src="nb_images/architecture.png" style="width:700px;height:400px;">
<div style="text-align:center;"><u><b> 图 2 </b></u>：YOLO<br> 的编码架构</div>

如果对象的中心/中点落入网格单元中，则该网格单元负责检测该对象。

由于你使用了 5 个锚框，因此每个 19 x19 单元格都编码有关 5 个框的信息。锚框仅由其宽度和高度定义。

为简单起见，你将展平形状 (19, 19, 5, 85) 编码的最后两个维度，因此 Deep CNN 的输出为 (19, 19, 425)。

<img src="nb_images/flatten.png" style="width:700px;height:400px;">
<div style="text-align:center;"><u><b> 图 3 </b></u>：展平最后两个维度<br></div>

#### 类别分数

现在，对于（每个单元格的）每个框，你将计算以下元素级乘积并提取该框包含特定类的概率。  
类别分数为 $score_{c,i} = p_{c} \times c_{i}$：存在对象 $p_{c}$ 的概率乘以该对象属于某个类别的概率 $c_{i}$。

<img src="nb_images/probability_extraction.png" style="width:700px;height:400px;">
<div style="text-align:center;"><u><b>图4</b></u>：查找每个框检测到的类别<br></div>

##### 图4示例
* 在图 4 中，假设对于框 1（单元格 1），对象存在的概率为 $p_{1}=0.60$。  因此，盒子 1（单元格 1）中存在对象的可能性为 60%。  
* 对象属于类别“类别 3（汽车）”的概率为 $c_{3}=0.73$。  
* 方框 1 和类别“3”的得分为 $score_{1,3}=0.60 \times 0.73 = 0.44$。  
* 假设你计算框 1 中所有 80 个类别的分数，发现汽车类别（类别 3）的分数是最大的。  因此，你将分数 0.44 和类别“3”分配给此框“1”。

#### 可视化类别
这是可视化 YOLO 在图像上预测内容的一种方法：

- 对于每个 19x19 网格单元，找到概率分数的最大值（在 80 个类别中取最大值，5 个锚框中的每一个都有一个最大值）。
- 根据网格单元认为最有可能的对象为该网格单元着色。

这样做会产生这张图片： 

<img src="nb_images/proba_map.png" style="width:300px;height:300px;">
<div style="text-align:center;"><u><b>图 5</b></u>：每个 19x19 网格单元根据哪一类具有最大预测概率进行着色单元.<br></div>

请注意，这种可视化并不是 YOLO 算法本身用于进行预测的核心部分；这只是可视化算法中间结果的好方法。 

#### 可视化边界框
可视化 YOLO 输出的另一种方法是绘制它输出的边界框。这样做会产生如下可视化效果：  

<img src="nb_images/anchor_map.png" style="width:200px;height:200px;">
<div style="text-align:center;"><u><b>图6</b></u>：每个单元格为你提供5个盒子。总的来说，模型只需看一次图像就可以预测：19x19x5 = 1805 个盒子（通过网络一个前向传播）！不同的颜色代表不同的类别。 <br></div>

#### 非极大值抑制
在上图中，唯一绘制的框是模型分配了高概率的框，但这仍然太多了。你希望将算法的输出减少到检测到的对象数量少得多。  

为此，你将使用 **非极大值抑制**。具体来说，你将执行以下步骤： 
- 摆脱低分盒子。这意味着，该盒子对于检测某个类别不是很有信心，要么是因为任何对象的概率都很低，要么是因为这个特定类别的概率很低。
- 当多个框相互重叠并检测到同一对象时，仅选择一个框。

<a name='2-2'></a>
### 2.2 - 根据类别分数阈值筛选

首先根据阈值筛选边界框，移除类别分数低于指定阈值的所有框。

该模型总共为你提供 19x19x5x85 个数字，每个框由 85 个数字描述。将 (19,19,5,85)（或 (19,19,425)）维张量重新排列为以下变量很方便：  
- `box_confidence`：形状为 $(19, 19, 5, 1)$ 的张量，包含 $p_c$（存在某个对象的置信概率），用于每个 19x19 单元格中预测的 5 个框。
- `boxes`：形状为 $(19, 19, 5, 4)$ 的张量，包含每个单元中 5 个盒子的中点和尺寸 $(b_x, b_y, b_h, b_w)$。
- `box_class_probs`：形状为 $(19, 19, 5, 80)$ 的张量，包含每个单元 5 个盒子的 80 个类中每个类的“类概率”$(c_1, c_2, ... c_{80})$。

<a name='ex-1'></a>
### 练习 1 - yolo_filter_boxes

实现`yolo_filter_boxes()`。
1. 通过进行元素乘积来计算框得分，如图 4 ($p \times c$) 所示。  
以下代码可以帮助你选择正确的运算符： 
```python
a = np.random.randn(19, 19, 5, 1)
b = np.random.randn(19, 19, 5, 80)
c = a * b # c 的形状将是 (19, 19, 5, 80)
```
这是**广播**（不同大小的向量相乘）的示例。

2. 对于每个框，找到：
    - 框得分最高的类别的索引
    - 对应的方框分数
    
    **有用的参考资料**
        * [tf.math.argmax](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/math/argmax)
        * [tf.math.reduce_max](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/math/reduce_max)

    **有用的提示**
        * 对于`argmax`和`reduce_max`的`axis`参数，如果要选择**最后**轴，一种方法是设置`axis=-1`。  这类似于 Python 数组索引，你可以使用 `arrayname[-1]` 选择数组的最后一个位置。
        * 应用 `reduce_max` 通常会折叠应用了最大值的轴。  `keepdims=False` 是默认选项，并允许删除该尺寸。  在此处应用最大值后，你无需保留最后一个尺寸。


3. 使用阈值创建蒙版。提醒一下：`([0.9, 0.3, 0.4, 0.5, 0.1] < 0.4)` 返回：`[False, True, False, False, True]`。你想要保留的盒子的掩码应为 `True`。 

4. 使用TensorFlow将遮罩应用到`box_class_scores`、`boxes`和`box_classes`上，过滤掉你不想要的盒子。你应该只留下你想要保留的盒子的子集。   

    **还有一个有用的参考**：
    * [tf.boolean_mask](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/boolean_mask)

   **还有一个有用的提示**：:) 
    * 对于`tf.boolean_mask`，可以保留默认的`axis=None`。

In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: yolo_filter_boxes

def yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = .6):
    """通过对对象和类置信度进行阈值化来过滤 YOLO 框。
    
    参数：
        boxes -- 形状为 (19, 19, 5, 4) 的张量
        box_confidence -- 形状为 (19, 19, 5, 1) 的张量
        box_class_probs -- 形状为 (19, 19, 5, 80) 的张量
        threshold -- 真实值，如果 [最高类别概率得分 < 阈值]，
                     然后去掉相应的框

    返回：
        scores -- 形状为(None,)的张量，包含所选框的类概率分数
        boxes -- 形状为 (None, 4) 的张量，包含所选框的 (b_x, b_y, b_h, b_w) 坐标
        classes -- 形状为 (None,) 的张量，包含所选框检测到的类别的索引

    注意：这里出现 "None" 是因为你不知道所选框的确切数量，因为它取决于阈值。 
    例如，如果有 10 个盒子，则实际输出的分数大小将为 (10,)。
    """
    
    ### START CODE HERE
    # 步骤 1：计算框分数
    ##（≈ 1 行）
    box_scores = None

    # 步骤2：使用最大值box_scores找到box_classes，记录相应的分数
    ##（≈ 2 行）
    # IMPORTANT：将轴设置为-1
    box_classes = None
    box_class_scores = None
    
    # 步骤3：使用"threshold"创建基于"box_class_scores"的过滤掩码。掩码应具有
    # 与 box_class_scores 相同的维度，并且对于要保留的框为 True（概率 >= 阈值）
    ## （≈ 1 行）
    filtering_mask = None
    
    # 步骤4：将掩码敷在box_class_scores、盒子和box_classes上
    ## （≈ 3 行）
    scores = None
    boxes = None
    classes = None
    ### END CODE HERE
    
    return scores, boxes, classes

In [ ]:
# 开始单元测试
tf.random.set_seed(10)
box_confidence = tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([19, 19, 5, 4], mean=1, stddev=4, seed = 1)
box_class_probs = tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = 0.5)

print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.shape))
print("boxes.shape = " + str(boxes.shape))
print("classes.shape = " + str(classes.shape))

assert type(scores) == EagerTensor, "Use tensorflow functions"
assert type(boxes) == EagerTensor, "Use tensorflow functions"
assert type(classes) == EagerTensor, "Use tensorflow functions"

assert scores.shape == (1789,), "Wrong shape in scores"
assert boxes.shape == (1789, 4), "Wrong shape in boxes"
assert classes.shape == (1789,), "Wrong shape in classes"

assert np.isclose(scores[2].numpy(), 9.270486), "Values are wrong on scores"
assert np.allclose(boxes[2].numpy(), [4.6399336, 3.2303846, 4.431282, -2.202031]), "Values are wrong on boxes"
assert classes[2].numpy() == 8, "Values are wrong on classes"

print("\033[92m All tests passed!")
# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>分数[2]</b>
        </td>
        <td>
           9.270486
        </td>
    </tr>
    <tr>
        <td>
            <b>盒子[2]</b>
        </td>
        <td>
           [ 4.6399336  3.2303846  4.431282  -2.202031 ]
        </td>
    </tr>
    <tr>
        <td>
            <b>类[2]</b>
        </td>
        <td>
           8
        </td>
    </tr>
        <tr>
        <td>
            <b>scores.shape</b>
        </td>
        <td>
           (1789,)
        </td>
    </tr>
    <tr>
        <td>
            <b>boxes.shape</b>
        </td>
        <td>
           (1789, 4)
        </td>
    </tr>
    <tr>
        <td>
            <b>classes.shape</b>
        </td>
        <td>
           (1789,)
        </td>
    </tr>

</table>

**注意** 在 `yolo_filter_boxes` 的测试中，你使用随机数来测试函数。  在实际数据中，`box_class_probs` 将包含 0 到 1 之间的非零概率值。  `boxes` 中的框坐标也将被选择，以便长度和高度都是非负的。

<a name='2-3'></a>
### 2.3 - 非极大值抑制

即使按类别分数阈值筛选后，仍可能保留大量重叠框。接下来使用非极大值抑制（NMS）进一步筛选边界框。

<img src="nb_images/non-max-suppression.png" style="width:500px;height:400px;">
<div style="text-align:center;"><u> <b>图7</b> </u>：在这个例子中，模型预测了3辆车，但实际上是同一辆车的3个预测。运行非极大值抑制 (NMS) 将仅选择 3 个框中最准确（概率最高）的一个。 <br></div>


非极大值抑制使用非常重要的函数，称为 **“交并比”**，或 IoU。
<img src="nb_images/iou.png" style="width:500px;height:400px;">
<div style="text-align:center;"><u> <b>图8</b> </u>：“交并比”的定义。 <br></div>

<a name='ex-2'></a>
### 练习 2 - iou

实现`iou()` 

一些提示：
- 此代码使用以下约定：(0,0) 是图像的左上角，(1,0) 是右上角，(1,1) 是右下角。换句话说，(0,0) 原点从图像的左上角开始。随着 x 的增加，你将向右移动。  随着 y 的增加，你会向下移动。
- 在本练习中，使用盒子的两个角来定义：左上角 $(x_1, y_1)$ 和右下角 $(x_2,y_2)$，而不是使用中点、高度和宽度。这使得计算交集变得更容易一些。
- 要计算矩形的面积，请将其高度 $(y_2 - y_1)$ 乘以宽度 $(x_2 - x_1)$。由于 $(x_1,y_1)$ 位于左上角，$x_2,y_2$ 位于右下角，因此这些差异应该是非负的。
- 找到两个盒子 $(xi_{1}, yi_{1}, xi_{2}, yi_{2})$ 的**交集**： 
    - 请随意在纸上画一些例子来从概念上阐明这一点。
    - 交点 $(xi_{1}, yi_{1})$ 的左上角是通过比较两个框的左上角 $(x_1, y_1)$ 并找到 x 坐标更接近右侧、y 坐标更接近底部的顶点来找到的。
    - 交集的右下角$(xi_{2}, yi_{2})$是通过比较两个盒子的右下角$(x_2,y_2)$并找到x坐标更接近左侧、y坐标更接近顶部的顶点来找到的。
    - 两个盒子**可能没有交集**。  如果你计算的交叉点坐标最终是交叉点框的右上角和/或左下角，你可以检测到这一点。  另一种思考方式是，如果你计算高度 $(y_2 - y_1)$ 或宽度 $(x_2 - x_1)$ 并发现这些长度中至少有一个为负，则不存在交集（交集面积为零）。  
    - 两个盒子可能在**边或顶点**相交，在这种情况下相交面积仍然为零。  当计算的交点的高度或宽度（或两者）为零时，就会发生这种情况。


**其他提示**

- `xi1` = 两个盒子的x1坐标的**max**imum
- `yi1` = 两个盒子的 y1 坐标的 **max**imum
- `xi2` = 两个盒子的x2坐标的**min**imum
- `yi2` = 两个盒子的 y2 坐标的 **min**imum
- `inter_area` = 你可以使用 `max(height, 0)` 和 `max(width, 0)`


In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: iou

def iou(box1, box2):
    """在box1和box2之间实现交并比（IoU）
    
    参数：
    box1 -- 第一个框，列出坐标为 (box1_x1, box1_y1, box1_x2, box_1_y2) 的对象
    box2 -- 第二个框，列出坐标为 (box2_x1, box2_y1, box2_x2, box2_y2) 的对象
    """


    (box1_x1, box1_y1, box1_x2, box1_y2) = box1
    (box2_x1, box2_y1, box2_x2, box2_y2) = box2

    ### START CODE HERE
    # 计算box1和box2交集的(yi1, xi1, yi2, xi2)坐标。计算它的面积。
    ##（约7行）
    xi1 = None
    yi1 = None
    xi2 = None
    yi2 = None
    inter_width = None
    inter_height =  None
    inter_area = None
    
    # 使用公式计算联合面积：Union(A,B) = A + B - Inter(A,B)
    ## （≈ 3 行）
    box1_area = None
    box2_area = None
    union_area = None
    
    # 计算 IoU
    iou = None
    ### END CODE HERE
    
    return iou

In [ ]:
# 开始单元测试
## 测试用例 1：盒子相交
box1 = (2, 1, 4, 3)
box2 = (1, 2, 3, 4)

print("iou for intersecting boxes = " + str(iou(box1, box2)))
assert iou(box1, box2) < 1, "The intersection area must be always smaller or equal than the union area."
assert np.isclose(iou(box1, box2), 0.14285714), "Wrong value. Check your implementation. Problem with intersecting boxes"

## 测试用例 2：盒子不相交
box1 = (1,2,3,4)
box2 = (5,6,7,8)
print("iou for non-intersecting boxes = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection must be 0"

## 测试用例 3：盒子仅在顶点相交
box1 = (1,1,2,2)
box2 = (2,2,3,3)
print("iou for boxes that only touch at vertices = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection at vertices must be 0"

## 测试用例 4：盒子仅在边缘相交
box1 = (1,1,3,3)
box2 = (2,3,3,4)
print("iou for boxes that only touch at edges = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection at edges must be 0"

print("\033[92m All tests passed!")
# 终端单元测试

**预期输出**：

```
iou for intersecting boxes = 0.14285714285714285
iou for non-intersecting boxes = 0.0
iou for boxes that only touch at vertices = 0.0
iou for boxes that only touch at edges = 0.0
```

<a name='2-4'></a>
### 2.4 - YOLO 非极大值抑制

你现在已准备好实现非极大值抑制。关键步骤如下：

1. **选择得分最高的框。**
2. **计算此框与同一类的所有其他框的重叠**。删除任何明显重叠的框 (IoU >= `iou_threshold`)。
3. **重复**该过程：返回步骤 1 并迭代，直到没有剩余框的分数低于当前所选框。

此过程有效地消除了与所选框显著重叠的框，只留下“最佳”候选框。

对于实现，请考虑使用分而治之的方法：

* **按类别划分**所有方框。
* 为每个类别单独申请非极大值抑制。
* 最后，**将**每个类别的结果合并到一组框中。

建议在即将进行的练习中使用这种方法，因为它简化了实现，同时确保了多个类别的准确性。

<a name='ex-3'></a>
### 练习 3 - yolo_non_max_suppression

使用 TensorFlow 实现 `yolo_non_max_suppression()`。 TensorFlow 有两个内置的函数用于实现非极大值抑制 （因此你实际上不需要使用 `iou()` 实现）：

**参考文档**： 

- [tf.image.non_max_suppression()](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/image/non_max_suppression)
```
tf.image.non_max_suppression(
    boxes, scores, max_output_size, iou_threshold=0.5,
    score_threshold=float('-inf'), name=None
)
```
**注意**：在下面的练习中，对于`tf.image.non_max_suppression()`，你只需设置`boxes`、`scores`、`max_output_size`和`iou_threshold`参数。


- [tf.gather()](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/gather)
```
tf.gather(
    params, indices, validate_indices=None, axis=None, batch_dims=0, name=None
)
```

**注意**：在下面的练习中，对于`tf.gather()`，你只需设置`params`和`indices`参数。

- [tf.boolean mask()](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/boolean_mask)
```
tf.boolean_mask(
    tensor, mask, axis=None, name='boolean_mask'
)
```

**注意**：在下面的练习中，对于`tf.boolean_mask()`，你只需设置`tensor`和`mask`参数。

- [tf.concat()](https://www.tensorflow.org/versions/r2.3/api_docs/python/tf/concat)
```
tf.concat(
    values, axis, name='concat'
)
```

In [ ]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: yolo_non_max_suppression

def yolo_non_max_suppression(scores, boxes, classes, max_boxes = 10, iou_threshold = 0.5):
    """
    将非极大值抑制 (NMS) 应用于一组盒子
    
    参数：
    scores -- 形状为 (None,) 的张量，yolo_filter_boxes() 的输出
    boxes -- 形状为 (None, 4) 的张量，yolo_filter_boxes() 的输出已缩放到图像大小（见下文）
    classes -- 形状为 (None,) 的张量，yolo_filter_boxes() 的输出
    max_boxes -- 整数，你想要的预测框的最大数量
    iou_threshold -- 真实值，"intersection over union" 用于NMS过滤的阈值
    
    返回：
    scores -- (None, ) 形状的张量，每个框的预测分数
    boxes -- 形状为 (None, 4) 的张量，预测的框坐标
    classes -- 形状为 (None, ) 的张量，每个框的预测类别
    
    注意：输出张量的 "None" 维度显然必须小于 max_boxes。另请注意，这
    函数将调换乐谱、盒子、类别的形状。这样做是为了方便。
    """
    boxes = tf.cast(boxes, dtype=tf.float32)
    scores = tf.cast(scores, dtype=tf.float32)

    nms_indices = []
    classes_labels = tf.unique(classes)[0] # 获得独特的课程
    
    for label in classes_labels:
        filtering_mask = classes == label
    
    #### START CODE HERE

        # 获取该课程的盒子
        # 将 tf.boolean_mask() 与 'boxes' 和 `filtering_mask` 一起使用
        boxes_label = None
        
        # 获取本课程的分数
        # 将 tf.boolean_mask() 与 'scores' 和 `filtering_mask` 一起使用
        scores_label = None
        
        if tf.shape(scores_label)[0] > 0:  # 检查是否有要处理的框
            
            # 使用 tf.image.non_max_suppression() 获取与你保留的框对应的索引列表
            ##（≈ 5 行）
            nms_indices_label = None(
                    None,
                    None,
                    None,
                    iou_threshold=None) 

            # 获取所选框的原始索引
            selected_indices = tf.squeeze(tf.where(filtering_mask), axis=1)
            
            # 将结果框附加到部分结果中
            # 将 tf.gather() 与 'selected_indices' 和 `nms_indices_label` 一起使用
            nms_indices.append(None)

    # 展平索引列表并连接
    # 将 tf.concat() 与 'nms_indices' 和 `axis=0` 一起使用
    nms_indices = None
    
    # 使用 tf.gather() 从分数、盒子和类别中仅选择 nms_indices
    ##（≈ 3 行）
    scores = None
    boxes = None
    classes = None
    
    ### END CODE HERE       
    
    # 按分数排序并返回最高的 max_boxes
    sort_order = tf.argsort(scores, direction='DESCENDING').numpy()
    scores = tf.gather(scores, sort_order[0:max_boxes])
    boxes = tf.gather(boxes, sort_order[0:max_boxes])
    classes = tf.gather(classes, sort_order[0:max_boxes])

    return scores, boxes, classes

In [ ]:
# 开始单元测试
# 此示例模仿幻灯片中显示的汽车与人重叠的情况
# 由于两个盒子属于不同的类别，因此它们永远不会受到抑制，尽管 iou_threshold
scores = np.array([0.855, 0.828])
boxes = np.array([[0.45, 0.2,  1.01, 2.6], [0.42, 0.15, 1.7, 1.01]])
classes = np.array([0, 1])

print(f"iou:    \t{iou(boxes[0], boxes[1])}")

scores2, boxes2, classes2 = yolo_non_max_suppression(scores, boxes, classes, iou_threshold = 0.9)

assert np.allclose(scores2.numpy(), [0.855, 0.828]), f"Wrong value on scores {scores2.numpy()}"
assert np.allclose(boxes2.numpy(), [[0.45, 0.2,  1.01, 2.6], [0.42, 0.15, 1.7, 1.01]]), f"Wrong value on boxes {boxes2.numpy()}"
assert np.array_equal(classes2.numpy(), [0, 1]), f"Wrong value on classes {classes2.numpy()}"

scores2, boxes2, classes2 = yolo_non_max_suppression(scores, boxes, classes, iou_threshold = 0.1)

assert np.allclose(scores2.numpy(), [0.855, 0.828]), f"Wrong value on scores {scores2.numpy()}"
assert np.allclose(boxes2.numpy(), [[0.45, 0.2,  1.01, 2.6], [0.42, 0.15, 1.7, 1.01]]), f"Wrong value on boxes {boxes2.numpy()}"
assert np.array_equal(classes2.numpy(), [0, 1]), f"Wrong value on classes {classes2.numpy()}"

classes = np.array([0, 0])

# 如果两个框属于同一类，则如果 iou 在 iou_threshold 下，则抑制一个框
scores2, boxes2, classes2 = yolo_non_max_suppression(scores, boxes, classes, iou_threshold = 0.15)

assert np.allclose(scores2.numpy(), [0.855]), f"Wrong value on scores {scores2.numpy()}"
assert np.allclose(boxes2.numpy(), [[0.45, 0.2,  1.01, 2.6]]), f"Wrong value on boxes {boxes2.numpy()}"
assert np.array_equal(classes2.numpy(), [0]), f"Wrong value on classes {classes2.numpy()}"

# 它必须返回两个盒子，因为它们属于不同的类
print(f"scores:  \t{scores2.numpy()}")
print(f"boxes:  \t{boxes2.numpy()}")     
print(f"classes:\t{classes2.numpy()}")

# 如果两个框属于同一类，则如果 iou 在 iou_threshold 下，则抑制一个框
scores2, boxes2, classes2 = yolo_non_max_suppression(scores, boxes, [0, 0], iou_threshold = 0.9)

assert np.allclose(scores2.numpy(), [0.855, 0.828]), f"Wrong value on scores {scores2.numpy()}"
assert np.allclose(boxes2.numpy(), [[0.45, 0.2,  1.01, 2.6], [0.42, 0.15, 1.7, 1.01]]), f"Wrong value on boxes {boxes2.numpy()}"
assert np.array_equal(classes2.numpy(), [0, 0]), f"Wrong value on classes {classes2.numpy()}"

from unit_tests import test_yolo_non_max_suppression

test_yolo_non_max_suppression(yolo_non_max_suppression)
# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>iou 盒子之间：</b>
        </td>
        <td>
            0.22780233025311375
        </td>
    </tr>
    <tr>
        <td>
            <b>分数</b>
        </td>
        <td>
           [0.855]
        </td>
    </tr>
    <tr>
        <td>
            <b> 盒子：</b>
        </td>
        <td>
           [[0.45 0.20 1.01 2.60 ]]
        </td>
    </tr>
        <tr>
        <td>
            <b>类：</b>
        </td>
        <td>
           [0]
        </td>
    </tr>
</table>

<a name='2-5'></a>
### 2.5 - 整合筛选过程

现在是时候实现函数了，它采用深度 CNN（19x19x5x85 维度编码）的输出，并使用刚刚实现的函数过滤所有框。 

<a name='ex-4'></a>
### 练习 4 - yolo_eval

实现 `yolo_eval()`，它获取 YOLO 编码的输出并使用分数阈值和 NMS 过滤框。你只需要了解最后一个实现细节。有几种表示盒子的方法，例如通过它们的角或通过它们的中点和高度/宽度。 YOLO 使用以下函数（已提供）在不同时间在几种此类格式之间进行转换： 

```python
boxes = yolo_boxes_to_corners(box_xy, box_wh) 
```
 将 yolo 框坐标 (x,y,w,h) 转换为框角坐标 (x1, y1, x2, y2) 以适合 `yolo_filter_boxes` 的输入
```python
boxes = scale_boxes(boxes, image_shape)
```
YOLO 的网络经过训练可以在 608x608 图像上运行。如果你在不同尺寸的图像上测试此数据（例如，汽车检测数据集具有 720x1280 图像），则此步骤会重新缩放框，以便可以将它们绘制在原始 720x1280 图像的顶部。  

别担心这两个函数；你将在下面看到需要在哪里调用它们。  

In [ ]:
def yolo_boxes_to_corners(box_xy, box_wh):
    """将 YOLO 框预测转换为边界框角点。"""
    box_mins = box_xy - (box_wh / 2.)
    box_maxes = box_xy + (box_wh / 2.)

    return tf.keras.backend.concatenate([
        box_mins[..., 1:2],  # y_min
        box_mins[..., 0:1],  # x_min
        box_maxes[..., 1:2],  # y_max
        box_maxes[..., 0:1]  # x_max
    ])


In [ ]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: yolo_eval

def yolo_eval(yolo_outputs, image_shape = (720, 1280), max_boxes=10, score_threshold=.6, iou_threshold=.5):
    """
    将 YOLO 编码的输出（很多框）转换为预测框及其分数、框坐标和类别。
    
    参数：
    yolo_outputs -- 编码模型的输出（对于 (608, 608, 3) 的 image_shape），包含 4 个张量：
                    box_xy: 形状为 (None, 19, 19, 5, 2) 的张量
                    box_wh: 形状为 (None, 19, 19, 5, 2) 的张量
                    box_confidence: 形状为 (None, 19, 19, 5, 1) 的张量
                    box_class_probs: 形状为 (None, 19, 19, 5, 80) 的张量
    image_shape -- 包含输入形状的形状为 (2,) 的张量，在这个 Notebook 中我们使用 (608., 608.) （必须是 float32 dtype）
    max_boxes -- 整数，你想要的预测框的最大数量
    score_threshold -- 真实值，如果[最高类概率得分<阈值]，则去掉对应的框
    iou_threshold -- 真实值，"intersection over union" 用于NMS过滤的阈值
    
    返回：
    scores -- (None, ) 形状的张量，每个框的预测分数
    boxes -- 形状为 (None, 4) 的张量，预测的框坐标
    classes -- 形状为 (None,) 的张量，每个框的预测类别
    """
    
    ### START CODE HERE
    # 检索 YOLO 模型的输出（约 1 行）
    box_xy, box_wh, box_confidence, box_class_probs = None
    
    # 转换框以准备过滤函数（将框 box_xy 和 box_wh 转换为角坐标）
    boxes = None
    
    # 使用你已实现的函数之一来执行阈值 score_threshold（约 1 行）的分数过滤
    scores, boxes, classes = None
    
    # 将框缩放回原始图像形状。
    boxes = None
    
    # 使用你已实现的函数之一来执行非极大值抑制 
    # 最大框数设置为 max_boxes，阈值设置为 iou_threshold（≈1 行）
    scores, boxes, classes = None
    ### END CODE HERE
    
    return scores, boxes, classes

In [ ]:
# 开始单元测试
tf.random.set_seed(10)
yolo_outputs = (tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1))
scores, boxes, classes = yolo_eval(yolo_outputs)
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

assert type(scores) == EagerTensor, "Use tensoflow functions"
assert type(boxes) == EagerTensor, "Use tensoflow functions"
assert type(classes) == EagerTensor, "Use tensoflow functions"

assert scores.shape == (10,), "Wrong shape"
assert boxes.shape == (10, 4), "Wrong shape"
assert classes.shape == (10,), "Wrong shape"
    
assert np.isclose(scores[2].numpy(), 171.60194), "Wrong value on scores"
assert np.allclose(boxes[2].numpy(), [-1240.3483, -3212.5881, -645.78, 2024.3052]), "Wrong value on boxes"
assert np.isclose(classes[2].numpy(), 16), "Wrong value on classes"
    
print("\033[92m All tests passed!")
# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>分数[2]</b>
        </td>
        <td>
           171.60194
        </td>
    </tr>
    <tr>
        <td>
            <b>盒子[2]</b>
        </td>
        <td>
           [-1240.3483 -3212.5881  -645.78    2024.3052]
        </td>
    </tr>
    <tr>
        <td>
            <b>类[2]</b>
        </td>
        <td>
           16
        </td>
    </tr> 
        <tr>
        <td>
            <b>scores.shape</b>
        </td>
        <td>
           (10,)
        </td>
    </tr>
    <tr>
        <td>
            <b>boxes.shape</b>
        </td>
        <td>
           (10, 4)
        </td>
    </tr>
    <tr>
        <td>
            <b>classes.shape</b>
        </td>
        <td>
           (10,)
        </td>
    </tr>

</table>

<a name='3'></a>
## 3 - 在图像上测试 YOLO 预训练模型

在本部分中，你将使用预训练模型并在汽车检测数据集上对其进行测试。  

<a name='3-1'></a>
### 3.1 - 定义类别、锚框和图像形状

你正在尝试检测 80 个类别，并使用 5 个锚框。有关 80 个类别和 5 个框的信息收集在两个文件中：“coco_classes.txt”和“yolo_anchors.txt”。你将从文本文件中读取类名和锚点。汽车检测数据集有 720x1280 图像，这些图像被预处理为 608x608 图像。

In [ ]:
class_names = read_classes("model_data/coco_classes.txt")
anchors = read_anchors("model_data/yolo_anchors.txt")
model_image_size = (608, 608) # 与 yolo_model 输入层大小相同

<a name='3-2'></a>
### 3.2 - 加载预训练模型

训练 YOLO 模型需要很长的时间，并且需要相当大的标记边界框的数据集，以适应大范围的目标类别。你将加载存储在“yolo.h5”中的现有预训练 Keras YOLO 模型。这些权重来自 YOLO 官方网站，并使用 Allan Zelener 编写的函数进行转换。参考文献位于 Notebook 的末尾。从技术上讲，这些是来自“YOLOv2”模型的参数，但在本 Notebook 中简称为“YOLO”。

运行下面的单元格以从此文件加载模型。

In [ ]:
yolo_model = load_model("model_data/", compile=False)

这会加载经过训练的 YOLO 模型的权重。以下是模型包含的层的摘要：

In [ ]:
yolo_model.summary()

**注意**：在某些计算机上，你可能会看到来自 Keras 的警告消息。如果你这样做了，别担心——这很好！

**提醒**：该模型将一批预处理后的输入图像（形状：(m, 608, 608, 3)）转换为形状（m, 19, 19, 5, 85）的张量，如图（2）所示。

<a name='3-3'></a>
### 3.3 - 将模型的输出转换为可用的边界框张量

`yolo_model` 的输出是一个 (m, 19, 19, 5, 85) 张量，需要经过重要的处理和转换。你需要调用 `yolo_head` 将从 `yolo_model` 获得的模型的编码格式化为可解读的内容：

yolo_model_outputs = yolo_model(image_data) 
yolo_outputs = yolo_head(yolo_model_outputs, 锚点, len(class_names))
变量 `yolo_outputs` 将被定义为一组 4 个张量，然后你可以将其用作 yolo_eval 函数的输入。如果你好奇yolo_head是如何实现的，你可以在文件`keras_yolo.py`中找到函数的定义。该文件也位于你的工作区中的以下路径：`yad2k/models/keras_yolo.py`。

<a name='3-4'></a>
### 3.4 - 过滤框

`yolo_outputs` 以正确的格式为你提供了 `yolo_model` 的所有预测框。要执行过滤并仅选择最佳框，你将调用你之前实现的 `yolo_eval` 来执行此操作：

    out_scores, out_boxes, out_classes = yolo_eval(yolo_outputs, [image.size[1],  image.size[0]], 10, 0.3, 0.5)

<a name='3-5'></a>
### 3.5 - 在图像上运行 YOLO

让乐趣开始吧！你将创建一个图表，总结如下：

`yolo_model.input` 被赋予 `yolo_model`。该模型用于计算输出 `yolo_model.output`
`yolo_model.output` 由 `yolo_head` 处理。它给你 `yolo_outputs`
`yolo_outputs` 经过过滤函数、`yolo_eval`。它输出你的预测：`out_scores`、`out_boxes`、`out_classes`。

现在，我们已经为你实现了 `predict(image_file)` 函数，它运行图来在图像上测试 YOLO，以计算 `out_scores`、`out_boxes`、`out_classes`。

下面的代码也使用以下函数：

    图像, image_data = preprocess_image("图像/" + image_file, model_image_size = (608, 608))
打开图像文件并缩放、重塑和标准化图像。它返回输出：

    图像：用于绘制方框的图像的 Python (PIL) 表示形式。你不需要使用它。
    image_data：表示图像的 NumPy 数组。这将是 CNN 的输入。

In [ ]:
def predict(image_file):
    """
    运行图表来预测 "image_file" 的框。打印并绘制预测结果。
    
    参数：
    image_file -- 存储在 "images" 文件夹中的图像的名称。
    
    返回：
    out_scores -- 形状张量 (None, )，预测框的分数
    out_boxes -- 形状为 (None, 4) 的张量，预测框的坐标
    out_classes -- (None, ) 形状的张量，预测框的类别索引
    
    注意："None" 实际上代表预测框的数量，它在 0 到 max_boxes 之间变化。 
    """

    # 预处理图像
    image, image_data = preprocess_image("images/" + image_file, model_image_size = (608, 608))
    
    yolo_model_outputs = yolo_model(image_data)
    yolo_outputs = yolo_head(yolo_model_outputs, anchors, len(class_names))
    
    out_scores, out_boxes, out_classes = yolo_eval(yolo_outputs, [image.size[1],  image.size[0]], 10, 0.3, 0.5)

    # 打印预测信息
    print('Found {} boxes for {}'.format(len(out_boxes), "images/" + image_file))
    # 生成用于绘制边界框的颜色。
    colors = get_colors_for_classes(len(class_names))
    # 在图像文件上绘制边界框
    #draw_boxes2(image, out_scores, out_boxes, out_classes, class_names, colors, image_shape)
    draw_boxes(image, out_boxes, out_classes, class_names, out_scores)
    # 将预测的边界框保存在图像上
    image.save(os.path.join("out", image_file), quality=100)
    # 将结果显示在Notebook中
    output_image = Image.open(os.path.join("out", image_file))
    imshow(output_image)

    return out_scores, out_boxes, out_classes

在“test.jpg”图像上运行以下单元格以验证你的函数是否正确。

In [ ]:
out_scores, out_boxes, out_classes = predict("test.jpg")

**预期输出**：

<table>
    <tr>
        <td>
            <b>找到10个图像框/test.jpg</b>
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.89 (367, 300) (745, 648)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.80 (761, 282) (942, 412)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.74 (159, 303) (346, 440)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
          0.70 (947, 324) (1280, 705)
        </td>
    </tr>
    <tr>
        <td>
            <b>公共汽车</b>
        </td>
        <td>
           0.67 (5, 266) (220, 407)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.66 (706, 279) (786, 350)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.60 (925, 285) (1045, 374)
        </td>
    </tr>
        <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.44 (336, 296) (378, 335)
        </td>
    </tr>
    <tr>
        <td>
            <b>汽车</b>
        </td>
        <td>
           0.37 (965, 273) (1022, 292)
        </td>
    </tr>
    <tr>
        <td>
            <b>交通灯</b>
        </td>
        <td>
           00.36 (681, 195) (692, 214)
        </td>
    </tr>
</table>

你刚刚运行的模型实际上能够检测“coco_classes.txt”中列出的 80 个不同的类别。要在你自己的图像上测试模型：
    1. 单击此 Notebook 上部栏中的“文件”，然后单击“打开”以转到你的 Coursera Hub。
    2. 将图像添加到 Jupyter Notebook 目录的“images”文件夹中
    3. 在代码上方的单元格中写下图像的名称
    4. 运行代码并查看算法的输出！

如果你要在 for 循环中对所有图像运行会话。你将得到以下结果：

<center>
<video width="400" height="200" src="nb_images/pred_video_compressed2.mp4" type="video/mp4" controls>
</video>
</center>

<div style="text-align:center;">YOLO 模型对在硅谷开车时从相机拍摄的照片进行的预测 <br> 感谢 <a href="https://www.drive.ai/">drive.ai</a> 提供此数据集！</div>

<a name='4'></a>
## 4 - YOLO 总结

- 输入图像 (608, 608, 3)
- 输入图像经过 CNN，产生 (19,19,5,85) 维输出。 
- 展平最后两个维度后，输出是形状为 (19, 19, 425) 的张量：
    - 输入图像上 19x19 网格中的每个单元格给出 425 个数字。 
    - 425 = 5 x 85 因为每个单元格包含 5 个框的预测，对应于 5 个锚框，如讲座中所示。 
    - 85 = 5 + 80 其中 5 是因为 $(p_c, b_x, b_y, b_h, b_w)$ 有 5 个数字，80 是我们要检测的类数
- 然后，你仅根据以下条件选择几个框：
    - 分数阈值：丢弃检测到分数低于阈值的类别的框
    - 非极大值抑制：计算交并比并避免选择重叠框
- 这给出了 YOLO 的最终输出。 

<div style="color:blue;">

**你应该记住什么**：
    
- YOLO 是一种最先进的目标检测模型，快速且准确
- 将输入图像传入 CNN，得到形状为 19x19x5x85 的输出张量。
- 编码可以看作是一个网格，其中每个 19x19 单元格包含有关 5 个框的信息。
- 你使用非极大值抑制过滤所有框。具体来说： 
    - 对检测类别的概率进行评分阈值，以仅保留准确（高概率）的框
    - 交并比 (IoU) 阈值消除重叠框
- 由于从随机初始化的权重训练 YOLO 模型并非易事，并且需要大型数据集和大量计算，因此本练习中使用了之前训练的模型参数。如果你愿意，你还可以使用自己的数据集尝试微调 YOLO 模型，尽管这将是一个相当重要的练习。

</div>

**恭喜！** 你已完成本作业。 

这是你所完成的所有工作的快速回顾。

你已经： 

- 汽车检测数据集中检测到的对象
- 实现非极大值抑制以实现更好的精度
- 将交并比实现为 NMS 的函数
- 根据模型的预测创建可用的边界框张量

了不起的工作！如果你想更多地了解这些想法的起源，请花一些时间阅读下面引用的论文。 

<a name='5'></a>
## 5 - 参考文献

Notebook 中提出的想法主要来自两篇 YOLO 论文。这里的实现也获得了重要的灵感，并使用了 Allan Zelener 的 GitHub 存储库中的许多组件。本次练习中使用的预训练权重来自YOLO官方网站。 
- 约瑟夫·雷德蒙、桑托什·迪瓦拉、罗斯·吉尔希克、阿里·法哈迪 - [你只看一次：统一、实时的物体检测](https://arxiv.org/abs/1506.02640) (2015)
- 约瑟夫·雷德蒙、阿里·法哈迪 - [YOLO9000：更好、更快、更强](https://arxiv.org/abs/1612.08242) (2016)
- 艾伦·泽莱纳 - [YAD2K：又一个暗网2 Keras](https://github.com/allanzelener/YAD2K)
- YOLO官方网站（https://pjreddie.com/darknet/yolo/） 

### 汽车检测数据集

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0px" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">Drive.ai 示例数据集</span>（由 drive.ai 提供）已获得许可<a rel="license" href="http://creativecommons.org/licenses/by/4.0/">知识共享归属4.0国际许可证</a>。感谢 Brody Huval、Chih Hu 和 Rahul Patel 提供此数据。 